# Lab 5A: Exploring Data Without Persistence

## 🎯 Learning Objectives

- Understand why and when to use DuckDB without persistence
- Practice inferring file types and schemas automatically
- Learn to shred nested JSON data
- Convert between data formats (CSV to Parquet)
- Query Parquet files directly without loading
- Query SQLite and other databases
- Work with Excel files in DuckDB

## 🎓 Conceptual Background

### Why Use a Database Without Persisting Data?

DuckDB's ability to query data files directly without creating persistent tables offers several advantages for exploratory analysis and data validation.

## 🚀 Step 1: Practice File Type Inference

Let's explore DuckDB's automatic file type detection.

In [ ]:
import duckdb

con = duckdb.connect(':memory:')

# Create sample files in different formats
con.execute("""
    CREATE TABLE sample AS
    SELECT 
        id,
        'Product ' || id as name,
        random() * 100 as price,
        '2023-01-01'::date + (random() * 365)::int as date
    FROM range(100)
""")

# Export to different formats
con.execute("COPY sample TO 'data/auto_test.csv' (FORMAT CSV, HEADER)")
con.execute("COPY sample TO 'data/auto_test.parquet' (FORMAT PARQUET)")
con.execute("COPY sample TO 'data/auto_test.json' (FORMAT JSON, ARRAY false)")

# Test automatic detection
csv_result = con.execute("SELECT COUNT(*) FROM 'data/auto_test.csv'").fetchone()
parquet_result = con.execute("SELECT COUNT(*) FROM 'data/auto_test.parquet'").fetchone()
json_result = con.execute("SELECT COUNT(*) FROM 'data/auto_test.json'").fetchone()

print(f"CSV: {csv_result[0]} rows")
print(f"Parquet: {parquet_result[0]} rows")
print(f"JSON: {json_result[0]} rows")

## 🚀 Step 2: Practice Schema Inference

Explore DuckDB's automatic schema detection.

In [ ]:
# Query a file and examine the inferred schema
con.execute("DESCRIBE SELECT * FROM 'data/sample/customers.csv'")
schema = con.fetchall()
print("Inferred schema for customers.csv:")
for column in schema:
    print(f"  {column[0]}: {column[1]}")

## 🚀 Step 3: Shred Nested JSON

Practice working with nested JSON structures.

In [ ]:
import json

# Create nested JSON sample
nested_data = [
    {
        "user": {"id": 1, "name": "Alice", "email": "alice@example.com"},
        "orders": [
            {"id": 101, "total": 100.50, "date": "2023-01-15"},
            {"id": 102, "total": 75.25, "date": "2023-02-20"}
        ]
    },
    {
        "user": {"id": 2, "name": "Bob", "email": "bob@example.com"},
        "orders": [
            {"id": 201, "total": 200.00, "date": "2023-01-10"}
        ]
    }
]

# Write nested JSON to file
with open('data/nested_orders.json', 'w') as f:
    json.dump(nested_data, f)

# Query nested JSON
result = con.execute("""
    SELECT 
        user->>'name' as customer_name,
        user->>'email' as customer_email,
        len(orders) as order_count,
        orders[1]->>'total' as first_order_total
    FROM read_json('data/nested_orders.json', format='auto')
""").fetchdf()
print(result)

## 🚀 Step 4: Convert CSV to Parquet

Practice format conversion and compare performance.

In [ ]:
import time
import os

# Read CSV and convert to Parquet
con.execute("""
    COPY (
        SELECT * FROM 'data/sample/customers.csv'
    ) TO 'data/customers.parquet' (FORMAT PARQUET)
""")

# Compare file sizes
csv_size = os.path.getsize('data/sample/customers.csv') / 1024  # KB
parquet_size = os.path.getsize('data/customers.parquet') / 1024  # KB

print(f"CSV size: {csv_size:.2f} KB")
print(f"Parquet size: {parquet_size:.2f} KB")
print(f"Compression ratio: {csv_size/parquet_size:.2f}x")

# Query performance comparison
start = time.time()
con.execute("SELECT COUNT(*) FROM 'data/sample/customers.csv'").fetchone()
csv_time = time.time() - start

start = time.time()
con.execute("SELECT COUNT(*) FROM 'data/customers.parquet'").fetchone()
parquet_time = time.time() - start

print(f"\nCSV query time: {csv_time:.4f}s")
print(f"Parquet query time: {parquet_time:.4f}s")
print(f"Speed improvement: {csv_time/parquet_time:.2f}x")

## 🚀 Step 5: Query Parquet Files Directly

Practice querying Parquet files without loading.

In [ ]:
# Query Parquet file directly
result = con.execute("""
    SELECT 
        segment,
        COUNT(*) as customer_count,
        AVG(loyalty_points) as avg_loyalty
    FROM 'data/customers.parquet'
    GROUP BY segment
""").fetchdf()
print("Direct Parquet query results:")
print(result)

## 🚀 Step 6: Query SQLite Databases

Practice querying other database formats.

In [ ]:
import sqlite3

# Create a SQLite database
sqlite_con = sqlite3.connect('data/test_sqlite.db')
sqlite_con.execute("""
    CREATE TABLE sqlite_products (
        id INTEGER PRIMARY KEY,
        name TEXT,
        price REAL
    )
""")
sqlite_con.execute("""
    INSERT INTO sqlite_products VALUES 
        (1, 'Product A', 10.99),
        (2, 'Product B', 20.50),
        (3, 'Product C', 15.75)
""")
sqlite_con.commit()
sqlite_con.close()

# Query SQLite from DuckDB
result = con.execute("""
    SELECT * FROM sqlite_scan('data/test_sqlite.db', 'sqlite_products')
""").fetchdf()
print("SQLite data queried from DuckDB:")
print(result)

## 🚀 Step 7: Work with Excel Files

Practice reading Excel files using pandas as intermediate step.

In [ ]:
# Create a sample Excel file
df = con.execute("SELECT * FROM sample LIMIT 10").df()
df.to_excel('data/sample_excel.xlsx', index=False)

# Read Excel via pandas and query with DuckDB
excel_df = pd.read_excel('data/sample_excel.xlsx')
con.register('excel_data', excel_df)
result = con.execute("SELECT * FROM excel_data WHERE price > 50").fetchdf()
print("Excel data queried via pandas:")
print(result)

## 💻 Exercise 1: Comprehensive File Format Comparison

Compare different file formats for a realistic dataset.

In [ ]:
# Your code here to compare CSV, Parquet, and JSON formats
# Include export time, file size, and query performance metrics

## 💻 Exercise 2: Complex JSON Shredding

Work with complex nested JSON structures.

In [ ]:
# Your code here to extract and flatten complex nested JSON
# Practice with multiple levels of nesting

## ✅ Verification

Verify your skills with this comprehensive test.

In [ ]:
print("=== Data Without Persistence Verification ===")

con = duckdb.connect(':memory:')

# Test 1: File type inference
con.execute("CREATE TABLE test AS SELECT * FROM range(100)")
con.execute("COPY test TO 'data/verify.csv' (FORMAT CSV, HEADER)")
csv_count = con.execute("SELECT COUNT(*) FROM 'data/verify.csv'").fetchone()[0]
print(f"✓ File type inference: {csv_count} == 100")

# Test 2: Schema inference
schema = con.execute("DESCRIBE SELECT * FROM 'data/verify.csv'").fetchall()
print(f"✓ Schema inference: {len(schema)} columns detected")

# Test 3: Format conversion
con.execute("COPY test TO 'data/verify.parquet' (FORMAT PARQUET)")
parquet_count = con.execute("SELECT COUNT(*) FROM 'data/verify.parquet'").fetchone()[0]
print(f"✓ Format conversion: {parquet_count} == 100")

# Test 4: File size comparison
csv_size = os.path.getsize('data/verify.csv')
parquet_size = os.path.getsize('data/verify.parquet')
print(f"✓ Parquet compression: {csv_size/parquet_size:.2f}x smaller")

print("=== All Verification Tests Passed ===")

## 📚 Next Steps

After completing this lab:

1. **Proceed to Lab 6**: Python Integration
2. **Practice more**: Experiment with different data formats
3. **Real-world data**: Apply these techniques to your own datasets